# 📘 Chapter 7 — Ensemble Learning and Random Forests

## Topic 1 — The Core Idea of Ensemble Learning

---

## A) Where to Read

In `7_Ensemble Learning and Random Forests.pdf`:
* Beginning of chapter — **"Ensemble Learning"**
* Subsection: **"Voting Classifiers"**
* The coin-flipping example
* Why a group of weak learners can outperform a single strong learner

---

## 🔷 What Is Ensemble Learning?

> **Definition:** Combining multiple models (predictors) to produce a better overall model.

Instead of **one expert** making a decision → use a **committee of experts voting**.

---

## 🔷 Why Does It Work?

**Key principle:** A group of weak learners can form a **strong learner**.

**Conditions required:**
1. Each model performs **slightly better than random**
2. Models make **uncorrelated errors**

**Result:**
* Majority voting **reduces variance**
* Errors **cancel out**
* Overall accuracy **improves**

> 📝 **Key idea:** If all models make the same mistakes, the ensemble gives no benefit. **Diversity** is the engine of ensemble learning.

---

## 🔷 Hard Voting vs Soft Voting

### 1️⃣ Hard Voting
Each classifier votes for a class. **Majority wins.**
```
Model A → Class 1
Model B → Class 1
Model C → Class 0
─────────────────
Final   → Class 1  (2 vs 1)
```

### 2️⃣ Soft Voting
Each classifier outputs **probabilities**. Average probabilities → choose highest.

> Soft voting usually performs **better** because it uses confidence information — a model that is 99% sure counts more than one that is 51% sure.

---

## 🧠 Mathematical Intuition (Very Important)

Assume:
* Each classifier has **51% accuracy**
* We combine **1,000 classifiers**
* Errors are independent

By the **Law of Large Numbers**, the probability that the majority vote is correct becomes **much higher than 51%**.

$$P(\text{majority correct}) \gg 0.51$$

**But there is a hard condition:**

> The classifiers must be **independent** (or at least not too correlated). If all models make the same mistakes → ensemble gives **no benefit**.

---

## 📝 Key Takeaways — Topic 1

| Concept | Core idea |
|---|---|
| Ensemble learning | Combine many models → better overall model |
| Why it works | Uncorrelated errors cancel out |
| Hard voting | Majority class wins |
| Soft voting | Average probabilities → uses confidence |
| Critical condition | Models must be **diverse** (uncorrelated errors) |
| Law of Large Numbers | Many weak independent learners → strong learner |

In [1]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# Create dataset
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Individual models
log_clf = LogisticRegression()
rf_clf = RandomForestClassifier()
svm_clf = SVC(probability=True)

# Ensemble
voting_clf = VotingClassifier(
    estimators=[
        ('lr', log_clf),
        ('rf', rf_clf),
        ('svc', svm_clf)
    ],
    voting='soft'   # try 'hard' too
)

voting_clf.fit(X_train, y_train)

print(voting_clf.score(X_test, y_test))

0.904


Important:

If using soft voting → SVC(probability=True) is required.

## ✅ Why Ensemble Accuracy Exceeds 60%

With 50 classifiers at 60% accuracy and mostly independent errors:
* Each classifier is **better than random**
* When we take majority vote → individual mistakes **cancel each other out**
* The probability that **more than half are wrong simultaneously** becomes very small

> The more independent models we add, the more sharply the error probability drops.

---

# 📘 Topic 2 — Bagging and Pasting

---

## A) Where to Read

In `7_Ensemble Learning and Random Forests.pdf`:
* Section: **"Bagging and Pasting"**
* Subsection: **Bootstrapping** explanation
* Look for: sampling **with replacement**
* Look for: **OOB (Out-of-Bag)** evaluation

---

## 🔷 What Is Bagging?

**Bagging = Bootstrap Aggregating**

Steps:
1. Take original training dataset
2. Create multiple random subsets **with replacement**
3. Train a model on each subset
4. Aggregate predictions (usually by **voting**)

---

## 🔷 What Is Bootstrapping?

**Sampling with replacement.**

Original dataset: `[1, 2, 3, 4, 5]`

A bootstrap sample might be: `[2, 2, 5, 1, 4]`

Notice:
* Some points **repeated**
* Some points **missing**

Each model sees a slightly different dataset → creates **diversity** → reduces **variance**.

---

## 🔷 Bagging vs Pasting

| Method | Sampling Type |
|---|---|
| **Bagging** | With replacement |
| **Pasting** | Without replacement |

> Bagging introduces more randomness — usually performs better.

---

## 🔷 Why Bagging Works

| Effect | Detail |
|---|---|
| ✅ Reduces variance | Each model trained on different data |
| ✅ Prevents overfitting | Aggregation smooths out noisy fits |
| ✅ Best with high-variance models | e.g., Decision Trees |
| ❌ Does NOT reduce bias | The base model's bias stays the same |

> 📝 **Key idea:** Bagging is a **variance reduction** technique. It works best when the base model overfits (high variance) — like an unconstrained Decision Tree.

---

## 🔷 Out-of-Bag (OOB) Evaluation

Because each model sees only **~63% of data** (due to bootstrapping), the remaining **~37% is never seen** by that model.

These unseen samples = **Out-of-Bag (OOB) instances**

> ✅ We can evaluate each model on its OOB instances → **free cross-validation** without needing a separate validation set.


---

## 📝 Key Takeaways — Topic 2

| Concept | Core idea |
|---|---|
| Bagging | Train many models on bootstrap samples → aggregate |
| Bootstrapping | Sampling **with** replacement → diversity |
| Pasting | Sampling **without** replacement |
| Why it works | Variance reduction through diversity |
| OOB evaluation | ~37% unseen data → free validation |
| Best base model | High-variance models (Decision Trees) |

In [2]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=500, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=100,
    bootstrap=True,        # This makes it bagging
    oob_score=True
)

bag_clf.fit(X_train, y_train)

print("Test accuracy:", bag_clf.score(X_test, y_test))
print("OOB score:", bag_clf.oob_score_)

Test accuracy: 0.904
OOB score: 0.9253333333333333


Important:

bootstrap=True → Bagging

bootstrap=False → Pasting

oob_score=True enables OOB evaluation

## ✅ Why Bagging Reduces Variance But Not Bias — Lock-In

**Variance reduction:**
* High-variance models change a lot with small data changes
* Bagging trains many models on slightly different datasets → different errors
* When averaged → **fluctuations cancel out** → more stable ensemble

**Why bias is not reduced:**
* Bias comes from **model assumptions and capacity limitations**
* If each tree is systematically wrong → averaging them doesn't fix the systematic error
* Bagging reduces **randomness** (variance), not **systematic error** (bias)

---

# 📘 Topic 3 — Random Forests

---

## A) Where to Read

In `7_Ensemble Learning and Random Forests.pdf`:
* Section: **"Random Forests"**
* Look for: **feature randomness**, `max_features`, comparison to pure bagging

---

## 🔷 What Is a Random Forest?

> **Random Forest = Bagging of Decision Trees + Random Feature Selection at each split**

---

## 🔷 The Key Difference from Bagging

| Method | Sample randomness | Feature randomness |
|---|---|---|
| **Bagging** | ✅ Random samples | ❌ All features considered at each split |
| **Random Forest** | ✅ Random samples | ✅ **Random subset of features** at each split |

**Example:** Dataset has 10 features → at each split, randomly select 3 → only those 3 are considered for the best split.

---

## 🔷 Why Add Feature Randomness?

**The problem with pure bagging:**
* If one feature is very strong → all trees split on it first → trees become **highly correlated** → averaging doesn't help much

**Feature randomness fixes this:**
* ✅ Makes trees more **independent**
* ✅ Reduces **correlation** between trees
* ✅ Further reduces **variance**

> 📝 **Key idea:** The magic of Random Forests is **decorrelation**. By randomly restricting which features each tree can use at each split, trees are forced to find different patterns — making their errors more independent and the ensemble more powerful.

---

## 🔷 Important Parameter: `max_features`

Controls how many features are randomly chosen at each split.

| `max_features` value | Effect |
|---|---|
| Lower | More randomness → lower variance, possibly higher bias |
| Higher | Less randomness → trees more similar → less benefit |

**Defaults:**
```python
# Classification (default)
max_features = "sqrt"   # sqrt(n_features)

# Regression (default)
max_features = 1.0      # all features (less randomness)
```

---

## 💻 Code Example
```python
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=500,      # number of trees
    max_features="sqrt",   # random feature subset per split
    oob_score=True,        # free OOB validation
    random_state=42,
    n_jobs=-1              # use all CPU cores
)
rf_clf.fit(X_train, y_train)
print("OOB Score:", rf_clf.oob_score_)
```

---

## 📝 Key Takeaways — Topic 3

| Concept | Core idea |
|---|---|
| Random Forest | Bagging + random feature subset at each split |
| Why better than pure bagging | Decorrelates trees → more variance reduction |
| `max_features` | Controls trade-off between randomness and expressiveness |
| OOB score | Still available — free validation |
| Best for | High-variance base learners (Decision Trees) |

In [3]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=500,
    max_features='sqrt',   # default for classification
    oob_score=True,
    random_state=42
)

rf_clf.fit(X_train, y_train)

print("Test accuracy:", rf_clf.score(X_test, y_test))
print("OOB score:", rf_clf.oob_score_)

Test accuracy: 0.888
OOB score: 0.904


Notice:
Random Forest is simpler to use than manual bagging.

## ✅ Why Decorrelation Is Critical — Lock-In

| Tree correlation | Effect on ensemble |
|---|---|
| **High** | Trees make same mistakes → errors don't cancel → ensemble ≈ one big tree |
| **Low** | Trees make different mistakes → errors cancel → variance drops significantly |

> 📝 **Key insight:** Ensemble improvement depends on two things: (1) individual model strength, and (2) **low correlation between models**. Feature randomness in Random Forests directly attacks correlation — this is what makes them powerful.

---

# 📘 Topic 4 — Extremely Randomized Trees (ExtraTrees)

---

## A) Where to Read

In `7_Ensemble Learning and Random Forests.pdf`:
* Subsection: **"Extremely Randomized Trees"**
* Difference between Random Forest and ExtraTrees
* **Random thresholds** at splits

---

## 🔷 What Are ExtraTrees?

> **ExtraTrees = Even more random than Random Forest**

---

## 🔷 Two Key Differences from Random Forest

| Property | Random Forest | ExtraTrees |
|---|---|---|
| Feature sampling | ✅ Random subset | ✅ Random subset |
| Split threshold | 🔍 **Searches for best** | 🎲 **Picks randomly** |

**In Random Forest:** for each selected feature → searches for the **optimal split threshold**

**In ExtraTrees:** picks a **random split threshold** → no search

---

## 🔷 Why Add Even More Randomness?

| Effect | Detail |
|---|---|
| ✅ Reduces variance further | More diverse trees → more error cancellation |
| ❌ Slightly increases bias | Random thresholds are not optimal |

> 📝 **Key idea:** The variance reduction often **outweighs** the bias increase — so ExtraTrees can match or outperform Random Forests, while training **much faster** (no threshold search needed).

---

## 🔷 Full Model Comparison

| Model | Data Sampling | Feature Sampling | Split Search |
|---|---|---|---|
| Decision Tree | ❌ No | ❌ No | Best split |
| Bagging | ✅ Yes | ❌ No | Best split |
| Random Forest | ✅ Yes | ✅ Yes | Best split |
| **ExtraTrees** | ✅ Yes | ✅ Yes | **Random split** |


---

## 📝 Key Takeaways — Topic 4

| Concept | Core idea |
|---|---|
| ExtraTrees vs Random Forest | Same feature randomness + **random thresholds** instead of optimal search |
| Effect on variance | ⬇ Reduces further |
| Effect on bias | ⬆ Increases slightly |
| Training speed | **Faster** — no threshold search |
| When to prefer | When speed matters or Random Forest overfits |

In [4]:
from sklearn.ensemble import ExtraTreesClassifier

extra_clf = ExtraTreesClassifier(
    n_estimators=500,
    random_state=42
)

extra_clf.fit(X_train, y_train)

print("Test accuracy:", extra_clf.score(X_test, y_test))

Test accuracy: 0.888


## ✅ Why Extra Randomness Can Improve Generalization — Lock-In

$$\text{Generalization Error} = \text{Bias}^2 + \text{Variance} + \text{Noise}$$

**ExtraTrees effect:**
* Slightly **increases bias** — random thresholds are not optimal
* Strongly **reduces variance** — trees are much less correlated

When variance was the dominant problem (deep trees overfit heavily):

$$\text{Variance reduction} > \text{Bias increase} \implies \text{better generalization}$$

> 📝 Deeper randomness also prevents trees from perfectly adapting to small dataset quirks — reducing memorization.

---

# 📘 Topic 5 — Feature Importance in Random Forests

---

## A) Where to Read

In `7_Ensemble Learning and Random Forests.pdf`:
* Section: **Feature Importance**
* How Random Forest computes it
* **Mean decrease in impurity** (Gini importance)

---

## 🔷 Why Random Forests Can Measure Feature Importance

Each time a feature is used to split a node:
* It reduces **impurity** (Gini or entropy)
* That reduction is **recorded**

Across all trees:
* We **average** the impurity reductions per feature
* Features that reduce impurity more → **more important**

---

## 🔷 How It Works Conceptually

For each feature:

1. Measure **impurity decrease** at every split where it's used
2. **Weight** by number of samples reaching that node
3. **Average** over all trees in the forest

$$\text{Importance}(f) = \frac{1}{N_{trees}} \sum_{trees} \sum_{\text{nodes using } f} \frac{m_{node}}{m} \cdot \Delta G$$

> Higher total impurity reduction → higher importance.


---

## 🔷 Important Limitation

Feature importance **can be biased** toward features with:
* Many categories (high cardinality)
* Continuous features with many possible thresholds

> ✅ Use feature importance for **guidance**, not as ground truth. For unbiased estimates, consider **permutation importance** instead.

---

## 📝 Key Takeaways — Topic 5

| Concept | Core idea |
|---|---|
| How computed | Average impurity reduction per feature across all trees |
| Weighting | By number of samples reaching each node |
| Higher importance | Feature reduces impurity more consistently |
| Limitation | Biased toward high-cardinality / continuous features |
| Accessed via | `rf_clf.feature_importances_` |

In [6]:
import numpy as np

importances = rf_clf.feature_importances_
# This returns normalized scores summing to 1.
for i, importance in enumerate(importances):
    print(f"Feature {i}: {importance:.4f}")

Feature 0: 0.4380
Feature 1: 0.5620


## ✅ Feature Importance Bias — Lock-In

> Features with **more possible splits** have more chances to produce a large impurity decrease **by chance** — even if they are not truly more predictive. This is a **multiple-testing effect**.

* **Continuous features** → infinite possible thresholds → higher probability of finding one that looks important
* **High-cardinality categoricals** → many partition combinations → same inflation effect

---

# 📘 Topic 6 — Boosting (The Big Shift in Philosophy)

---

## A) Where to Read

In `7_Ensemble Learning and Random Forests.pdf`:
* Section: **"Boosting"**
* Subsections: **AdaBoost**, **Gradient Boosting**
* The introductory part explaining the **sequential nature** of boosting

---

## 🔷 What Is Boosting?

> **Boosting builds models sequentially, not independently.**

Each new model **focuses on correcting the mistakes** of the previous models.

---

## 🔷 Core Difference from Bagging

| | Bagging | Boosting |
|---|---|---|
| Training order | **Independent** (parallel) | **Sequential** |
| Primary effect | Reduces **variance** | Reduces **bias** |
| Combination | Simple averaging | **Weighted** combination |
| Focus | All samples equally | **Hard cases** emphasized |

---

## 🔷 High-Level Intuition

Instead of: *"Many trees vote equally"*

Boosting does:
```
1. Train first weak learner on original data
2. Identify mistakes
3. Increase importance of misclassified points
4. Train next learner focusing on hard cases
5. Repeat N times
6. Final prediction = weighted sum of all learners
```

---

## 🔷 Why Sequential Training Reduces Bias

* Each individual learner is **weak** (e.g., shallow tree / "stump")
* But each learner **corrects the residual error** of the previous one
* The ensemble progressively **fits harder and harder cases**
* Over many rounds → **bias drops significantly**

> 📝 **Key idea:** Bagging builds diverse models to reduce variance. Boosting builds **corrective** models to reduce bias. They are philosophically opposite approaches to the same problem.

---

## 📝 Key Takeaways — Topic 6

| Concept | Core idea |
|---|---|
| Boosting | Sequential training — each model corrects previous errors |
| Reduces | **Bias** (not primarily variance) |
| Combination | **Weighted** sum (not simple average) |
| Risk | Can **overfit** if too many rounds (unlike bagging) |
| Key algorithms | **AdaBoost**, **Gradient Boosting**, XGBoost, LightGBM |

## ✅ Boosting Reduces Bias — Lock-In

> **Boosting** trains models sequentially, each correcting the systematic errors of the previous → progressively increases model complexity → **reduces bias**
>
> **Bagging** averages independent models to stabilize predictions → does NOT actively fix systematic mistakes → **reduces variance**

---

# 📘 Topic 7 — AdaBoost (Adaptive Boosting)

---

## A) Where to Read

In `7_Ensemble Learning and Random Forests.pdf`:
* Subsection: **AdaBoost**
* Look for: sample weight updating, weak learners, weighted voting

---

## 🔷 Core Idea of AdaBoost

1. Train a weak learner (usually a **shallow tree / stump**)
2. **Increase weights** of misclassified samples
3. Train next learner on **weighted data**
4. Repeat
5. Final prediction = **weighted vote** of all learners

---

## 🔷 Step-by-Step Mechanism

### Step 1: Initialize Weights
All training samples start with **equal weight**:

$$w_i = \frac{1}{m}$$

### Step 2: Train Weak Learner + Compute Weighted Error

$$\text{error} = \sum_{i} w_i \cdot \mathbb{1}(y_i \neq \hat{y}_i)$$

### Step 3: Compute Learner Weight ($\alpha$)

$$\alpha = \eta \cdot \ln\left(\frac{1 - \text{error}}{\text{error}}\right)$$

| Error | $\alpha$ | Meaning |
|---|---|---|
| Low error | Large $\alpha$ | Strong learner → high vote weight |
| High error | Small $\alpha$ | Weak learner → low vote weight |
| error = 0.5 | $\alpha = 0$ | Random → no vote weight |

### Step 4: Update Sample Weights

* **Misclassified** samples → **weight increases** (focus on hard cases)
* **Correctly classified** → weight decreases

This forces the next learner to focus on the examples that were hardest.

---

## 🔷 Final Prediction

$$\text{Final prediction} = \text{sign}\left(\sum_t \alpha_t h_t(\mathbf{x})\right)$$

A **weighted majority vote** across all learners.


---

## 📝 Key Takeaways — Topic 7

| Concept | Core idea |
|---|---|
| Weak learner | Usually a **decision stump** (max_depth=1) |
| Weight update | Misclassified → higher weight → next model focuses there |
| $\alpha$ | Learner vote weight — proportional to accuracy |
| Final output | **Weighted** majority vote |
| Primarily reduces | **Bias** |
| Risk | Too many estimators → can **overfit** |
| `learning_rate` | Shrinks each $\alpha$ → controls contribution of each learner |

In [7]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1),
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)

ada_clf.fit(X_train, y_train)

print("Test accuracy:", ada_clf.score(X_test, y_test))

Test accuracy: 0.896


Important:

max_depth=1 → decision stump

That’s classic AdaBoost setup

## ✅ AdaBoost: Why error < 0.5 Is Required — Lock-In

$$\alpha = \eta \cdot \ln\left(\frac{1 - \text{error}}{\text{error}}\right)$$

| Case | $\alpha$ | Effect |
|---|---|---|
| error $< 0.5$ | $\alpha > 0$ | ✅ Positive vote — contributes correctly |
| error $= 0.5$ | $\alpha = 0$ | ❌ Zero weight — contributes nothing |
| error $> 0.5$ | $\alpha < 0$ | ❌ Vote **flips** — hurts the ensemble |

> 📝 **Every weak learner must perform better than random guessing** — otherwise its vote weight becomes zero or negative.

---

# 📘 Topic 8 — Gradient Boosting

---

## A) Where to Read

In `7_Ensemble Learning and Random Forests.pdf`:
* Subsection: **Gradient Boosting**
* Look for: residual fitting, sequential correction, learning rate, shrinkage, early stopping

---

## 🔷 Core Idea of Gradient Boosting

> Instead of modifying sample weights (like AdaBoost), Gradient Boosting **trains each new model to predict the residual errors** of the previous model.

---

## 🔷 Step-by-Step Process (Regression Example)

**Step 1:** Train first model:
$$F_1(\mathbf{x}) = h_1(\mathbf{x})$$

**Step 2:** Compute residuals:
$$r_i = y_i - F_1(\mathbf{x}_i)$$

**Step 3:** Train second model to predict residuals:
$$h_2(\mathbf{x}) \approx \mathbf{r}$$

Update the ensemble:
$$F_2(\mathbf{x}) = F_1(\mathbf{x}) + \eta \cdot h_2(\mathbf{x})$$

**Step 4:** Repeat — each new model corrects previous model's residual errors.

---

## 🔷 Why It's Called "Gradient" Boosting

> It performs **gradient descent in function space**.

Instead of optimizing parameters directly, it adds new models in the direction of the **negative gradient of the loss function**.

* For regression (MSE loss) → residuals = negative gradient → fitting residuals **is** gradient descent
* For other losses → fit the generalized negative gradient

---

## 🔷 Learning Rate ($\eta$) — Shrinkage

Controls the **step size** of each new model's contribution.

| $\eta$ value | Effect | Risk |
|---|---|---|
| **Small** | Slower learning, more trees needed | Better generalization |
| **Large** | Faster learning | Overfitting |

> 📝 **Key idea:** Small $\eta$ + more trees almost always beats large $\eta$ + fewer trees. This is called **shrinkage**.

---

## 📝 Key Takeaways — Topic 8

| Concept | Core idea |
|---|---|
| Core mechanism | Each model fits **residuals** of previous ensemble |
| "Gradient" | Gradient descent in function space |
| Learning rate $\eta$ | Controls contribution of each tree — smaller = better generalization |
| Weak learner | Shallow trees (typically `max_depth=3–5`) |
| Primarily reduces | **Bias** (like all boosting) |
| Risk | Overfitting with too many trees or large $\eta$ |
| Fix for overfitting | Early stopping + small $\eta$ |

---

## 🔷 AdaBoost vs Gradient Boosting

| | AdaBoost | Gradient Boosting |
|---|---|---|
| Correction mechanism | **Re-weights samples** | **Fits residuals** |
| Weak learner | Decision stump (depth=1) | Shallow tree (depth=3–5) |
| Flexibility | Limited to classification | Works for **any differentiable loss** |
| Modern usage | Less common | **Foundation of XGBoost, LightGBM** |

In [8]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_clf.fit(X_train, y_train)

print("Test accuracy:", gb_clf.score(X_test, y_test))

Test accuracy: 0.88


## ✅ Small Learning Rate — Lock-In

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x})$$

> A small learning rate improves generalization because it makes the boosting process take **smaller, more controlled steps in function space**, preventing overfitting and acting as a regularizer — though it requires **more trees to converge**.

| $\eta$ | Steps | Trees needed | Generalization |
|---|---|---|---|
| Large | Big jumps → fits noise | Few | Worse |
| Small | Gradual refinement | More | **Better** |

---

# 📘 Final Topic — Early Stopping in Gradient Boosting

---

## A) Where to Read

In `7_Ensemble Learning and Random Forests.pdf`:
* Subsection: **Early Stopping**
* Look for: validation error monitoring, `staged_predict()`

---

## 🔷 What Is Early Stopping?

Instead of training a **fixed number of trees**:

1. Train incrementally
2. Evaluate **validation error** after each tree
3. **Stop** when validation error starts increasing

---

## 🔷 Why It Works

Boosting can overfit if too many trees are added.

Early stopping:
* ✅ Finds **optimal number of trees**
* ✅ Prevents overfitting
* ✅ Improves generalization

---

## 🔷 Conceptual View

| Metric | Behavior over trees |
|---|---|
| Training error | Always **decreases** |
| Validation error | Decreases → **minimum** → increases (overfitting) |

> Stop at **minimum validation error**.

---

## 📝 Key Takeaways — Early Stopping

| Concept | Core idea |
|---|---|
| Problem it solves | Too many trees → overfitting |
| Mechanism | Monitor validation error per tree → stop at minimum |
| `staged_predict()` | Returns predictions after each tree → find best $n$ |
| Combined with small $\eta$ | **Best practice** — slow learning + early stop = strong generalization |

> 📝 **Key idea:** Early stopping and small learning rate work together. Small $\eta$ creates a smooth error curve that makes the optimal stopping point easier to find reliably.

## ✅ Training vs Validation Error in Gradient Boosting — Lock-In

| Stage | Bias | Variance | Training Error | Validation Error |
|---|---|---|---|---|
| Early (few trees) | High | Low | High | High |
| Optimal | Balanced | Balanced | Low | **Minimum** ✅ |
| Too many trees | Low | **High** | Still decreasing | **Increasing** ❌ |

> 📝 **Core insight:**
> * Training error **always decreases** — each new tree explicitly minimizes training loss
> * Validation error **eventually increases** — additional trees fit noise → variance dominates → overfitting

$$\text{Generalization Error} = \underbrace{\text{Bias}^2}_{\text{↓ always}} + \underbrace{\text{Variance}}_{\text{↑ eventually}} + \text{Noise}$$

---

# 🏆 Chapter 7 — Full Mastery Achieved

| Topic | Core idea |
|---|---|
| **Ensemble intuition** | Diverse weak learners → strong learner via error cancellation |
| **Bagging & Pasting** | Bootstrap samples → variance reduction + free OOB validation |
| **Random Forests** | Bagging + random feature subsets → decorrelation → lower variance |
| **ExtraTrees** | Random thresholds → even lower variance, faster training |
| **Feature Importance** | Mean impurity decrease — biased toward high-cardinality features |
| **Boosting philosophy** | Sequential correction → bias reduction |
| **AdaBoost** | Re-weight misclassified samples → $\alpha$ weighted vote |
| **Gradient Boosting** | Fit residuals → gradient descent in function space |
| **Learning rate** | Small $\eta$ = shrinkage = regularization → better generalization |
| **Early stopping** | Stop at minimum validation error → optimal tree count |

---

## 🧠 Unifying Thread

$$\underbrace{\text{Bagging / RF}}_{\text{reduce variance}} \longleftrightarrow \underbrace{\text{Boosting}}_{\text{reduce bias}} \longrightarrow \underbrace{\text{Early stopping + small } \eta}_{\text{control variance in boosting}}$$

> **Chapter 7 in one sentence:** Ensemble methods combine weak learners to build strong ones — either by **averaging diverse models** (bagging/RF) to reduce variance, or by **sequentially correcting errors** (boosting) to reduce bias, with the two approaches attacking the bias–variance tradeoff from opposite directions.

# 📘 Chapter 7 — Ensemble Learning and Random Forests: Full Mastery Summary

---

## Core Theme
> Combining multiple weak learners to build a powerful model.

---

## 1️⃣ What Is Ensemble Learning?

**Definition:** Combines multiple models to produce better predictive performance than any single model.

**Core Principle:** A group of weak learners can form a strong learner — if they are **diverse** and **slightly better than random**.

**Why it works:** Errors cancel out → variance reduces → stability improves.

| Type | Structure |
|---|---|
| Voting | Parallel models |
| Boosting | Sequential models |

---

## 2️⃣ Voting Classifiers

| Type | Mechanism |
|---|---|
| **Hard Voting** | Majority class wins |
| **Soft Voting** | Average probabilities → usually better |

**Key condition for success:** Models must be individually competent and have **low correlation between errors**.

---

## 3️⃣ Bagging (Bootstrap Aggregating)

Train multiple models on **random subsets with replacement**, then average predictions.

* Each model sees **~63%** of training samples
* Leaves **~37% unused** (OOB samples)
* ✅ Reduces **variance** | ❌ Does not reduce bias

---

## 4️⃣ Out-of-Bag (OOB) Evaluation

~37% of data unseen by each model → acts as **free cross-validation**.

No separate validation set required.

---

## 5️⃣ Random Forests

> **Bagging of Decision Trees + Random feature selection at each split**

At each node: only a **random subset of features** is considered.

$$\text{max\_features} = \sqrt{n\_features} \quad \text{(default, classification)}$$

**Why feature randomness?** Reduces **correlation between trees** → better ensemble performance.

* ✅ Strongly reduces **variance**
* ✅ Slight bias reduction vs single tree

---

## 6️⃣ ExtraTrees

Even more random than Random Forest:
* Random feature subset ✅
* **Random split thresholds** (no optimal search) ✅

| Effect | Direction |
|---|---|
| Bias | ⬆ Slightly higher |
| Variance | ⬇ Lower |
| Training speed | **Faster** |

---

## 7️⃣ Feature Importance

Computed via **mean decrease in impurity** — averaged across all trees.

**Limitation:** Biased toward continuous / high-cardinality features — more split possibilities → higher chance of large impurity decrease **by chance**.

---

## 8️⃣ Boosting — The Philosophical Shift

| | Bagging | Boosting |
|---|---|---|
| Model order | Independent | **Sequential** |
| Primary effect | Reduces variance | Reduces **bias** |
| Combination | Averaging | Weighted additive |

> Each new model **corrects the errors** of previous models.

---

## 9️⃣ AdaBoost

**Core mechanism:** Re-weight misclassified samples → train next learner on harder cases.

$$\alpha = \eta \ln\left(\frac{1 - \text{error}}{\text{error}}\right)$$

**Critical condition:** Error must be **< 0.5** — otherwise $\alpha \leq 0$ and the learner contributes nothing or hurts.

* ✅ Reduces **bias** | ❗ Can increase variance with too many estimators

---

## 🔟 Gradient Boosting

> Each new model predicts the **residual errors** of the previous model.

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x})$$

**Interpretation:** Gradient descent in **function space**.

| $\eta$ | Effect |
|---|---|
| Small | Slow learning, better generalization, more trees needed |
| Large | Fast learning, higher overfitting risk |

**Early Stopping:** Stop at minimum validation error — training error always decreases, validation error eventually rises.

---

## 🧠 Bias–Variance Perspective

| Method | Bias | Variance |
|---|---|---|
| Single Tree | Low | **High** |
| Bagging | Same | Lower |
| Random Forest | Slightly higher | **Much lower** |
| Boosting | **Lower** | Can increase |
| Gradient Boosting | **Very low** | Needs regularization |

---

## 🔥 If You Remember Only 5 Things

1. Ensemble improvement = **strength + diversity**
2. Bagging → reduces **variance**
3. Random Forest = Bagging + **feature randomness**
4. Boosting → reduces **bias** sequentially
5. Small $\eta$ + early stopping = **strong generalization**

---

## 🎯 Final Mental Model

$$\underbrace{\text{Bagging / RF}}_{\text{diversity via sampling}} \quad \underbrace{\text{Boosting}}_{\text{diversity via correction}} \quad \underbrace{\text{small } \eta + \text{early stop}}_{\text{balance both}}$$

> **Chapter 7 in one sentence:** Ensemble methods beat single models by combining **strength** and **diversity** — either averaging diverse models to reduce variance (Bagging/RF), or sequentially correcting errors to reduce bias (Boosting), with regularization techniques balancing both.

# 📘 Chapter 7 — Ensemble Learning: Exercises Q&A

---

**Q1. You have five models all achieving 95% precision on the same training data. Can combining them get better results?**

**Yes — possibly**, but only if the models make **different errors**.

* If all five produce **highly correlated predictions** → combining them will not help much
* If they are **different algorithms** with uncorrelated errors → voting/averaging can reduce variance and improve overall performance

> ✅ The key condition: **diversity of errors**, not diversity of accuracy.

---

**Q2. What is the difference between hard and soft voting classifiers?**

| | Hard Voting | Soft Voting |
|---|---|---|
| Mechanism | Majority class wins | Average predicted **probabilities** |
| Uses confidence | ❌ No | ✅ Yes |
| Usually better | — | ✅ Soft voting |

> Soft voting performs better because a model that is 99% confident counts more than one that is 51% confident.

---

**Q3. Which ensemble methods can be parallelized across multiple servers?**

| Method | Parallelizable? | Why |
|---|---|---|
| Bagging | ✅ Yes | Models trained independently |
| Pasting | ✅ Yes | Models trained independently |
| Random Forests | ✅ Yes | Trees trained independently |
| Stacking | ✅ Yes (per layer) | Base models independent |
| **Boosting** | ❌ No | Each model depends on previous model's errors |

---

**Q4. What is the benefit of out-of-bag (OOB) evaluation?**

OOB evaluation provides a **built-in validation method** for bagging-based ensembles:
* Each bootstrap sample leaves ~37% of data unseen
* Those unseen samples act as a **free validation set**
* No separate validation set required → more data available for training

---

**Q5. What makes Extra-Trees more random than Random Forests? Faster or slower?**

| Property | Random Forest | Extra-Trees |
|---|---|---|
| Feature sampling | ✅ Random subset | ✅ Random subset |
| Split threshold | 🔍 Searches for best | 🎲 **Chosen randomly** |
| Speed | Slower | **Faster** |
| Variance | Lower | Even lower |
| Bias | Lower | Slightly higher |

> Extra randomness reduces tree correlation → lower variance → can improve generalization. Faster because it skips the optimal split search.

---

**Q6. Your AdaBoost ensemble underfits. What hyperparameters should you tweak?**

Underfitting = **high bias** → increase model capacity:

* ↑ `n_estimators` — more learners to correct residual errors
* ↑ `learning_rate` — each learner contributes more
* Use a **more complex base learner** (e.g., deeper trees instead of stumps)

> All three increase the expressiveness of the ensemble and reduce bias.

---

**Q7. Your Gradient Boosting ensemble overfits. Should you increase or decrease the learning rate?**

**Decrease the learning rate.**

* Overfitting = **high variance** from overly aggressive updates
* Lowering $\eta$ → smaller, more controlled steps → better generalization

> Usually combined with: reducing `max_depth`, reducing `n_estimators`, or using early stopping.

8. Load the MNIST data (introduced in Chapter 3), and split it into a training set, a
validation set, and a test set (e.g., use 50,000 instances for training, 10,000 for val‐
idation, and 10,000 for testing). Then train various classifiers, such as a Random
Forest classifier, an Extra-Trees classifier, and an SVM. Next, try to combine
them into an ensemble that outperforms them all on the validation set, using a
soft or hard voting classifier. Once you have found one, try it on the test set. How
much better does it perform compared to the individual classifiers?

In [9]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1) Load MNIST
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(np.uint8)

# 2) Split: 50k train, 10k val, 10k test (from the classic 70k MNIST set)
# Shuffle once, then slice to fixed sizes.
rng = np.random.RandomState(42)
idx = rng.permutation(len(X))
X, y = X[idx], y[idx]

X_train, y_train = X[:50_000], y[:50_000]
X_val,   y_val   = X[50_000:60_000], y[50_000:60_000]
X_test,  y_test  = X[60_000:70_000], y[60_000:70_000]

# 3) Train individual classifiers
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
et = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42)
svm = SVC(kernel="rbf", C=5, gamma="scale", probability=True, random_state=42)  # probability=True needed for soft voting

models = [("rf", rf), ("et", et), ("svm", svm)]

for name, clf in models:
    clf.fit(X_train, y_train)

def eval_model(clf, Xv, yv, Xt, yt, name="model"):
    val_acc = accuracy_score(yv, clf.predict(Xv))
    test_acc = accuracy_score(yt, clf.predict(Xt))
    print(f"{name:>10} | val acc: {val_acc:.5f} | test acc: {test_acc:.5f}")
    return val_acc, test_acc

print("Individual models:")
rf_val, rf_test = eval_model(rf, X_val, y_val, X_test, y_test, "RandomForest")
et_val, et_test = eval_model(et, X_val, y_val, X_test, y_test, "ExtraTrees")
sv_val, sv_test = eval_model(svm, X_val, y_val, X_test, y_test, "SVM (RBF)")

# 4) Combine into an ensemble (soft voting usually best if probabilities are decent)
voting_soft = VotingClassifier(
    estimators=models,
    voting="soft",
    n_jobs=-1
)
voting_soft.fit(X_train, y_train)

print("\nEnsemble:")
ens_val, ens_test = eval_model(voting_soft, X_val, y_val, X_test, y_test, "SoftVoting")

# 5) Compare improvements vs best individual model
best_ind_val = max(rf_val, et_val, sv_val)
best_ind_test = max(rf_test, et_test, sv_test)
print("\nImprovement vs best individual:")
print(f"Validation: {ens_val - best_ind_val:+.5f}")
print(f"Test:       {ens_test - best_ind_test:+.5f}")

Individual models:
RandomForest | val acc: 0.96960 | test acc: 0.96940
ExtraTrees | val acc: 0.97220 | test acc: 0.97240
 SVM (RBF) | val acc: 0.98220 | test acc: 0.98250

Ensemble:
SoftVoting | val acc: 0.98310 | test acc: 0.98150

Improvement vs best individual:
Validation: +0.00090
Test:       -0.00100


9. Run the individual classifiers from the previous exercise to make predictions on
the validation set, and create a new training set with the resulting predictions:
each training instance is a vector containing the set of predictions from all your
classifiers for an image, and the target is the image’s class. Train a classifier on
this new training set. Congratulations, you have just trained a blender, and
together with the classifiers they form a stacking ensemble! Now let’s evaluate the
ensemble on the test set. For each image in the test set, make predictions with all
your classifiers, then feed the predictions to the blender to get the ensemble’s pre‐
dictions. How does it compare to the voting classifier you trained earlier?

In [10]:
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

# Assumes you already ran Exercise 8 and have:
# X_train, y_train, X_val, y_val, X_test, y_test
# and fitted base models: rf, et, svm
# plus (optionally) voting_soft for comparison.

base_models = [rf, et, svm]

# 1) Build the "meta" (stacking) training set using VALIDATION predictions
#    Each row = [pred_rf, pred_et, pred_svm]
val_meta_X = np.column_stack([m.predict(X_val) for m in base_models])
val_meta_y = y_val

# 2) Train the blender (meta-model)
# Logistic Regression is a common, strong default for stacking on class-label features
blender = LogisticRegression(max_iter=2000, n_jobs=-1, multi_class="auto")
blender.fit(val_meta_X, val_meta_y)

# 3) Evaluate stacking on TEST:
#    Create meta-features from TEST predictions of base models, then blender predicts
test_meta_X = np.column_stack([m.predict(X_test) for m in base_models])
stack_pred = blender.predict(test_meta_X)
stack_test_acc = accuracy_score(y_test, stack_pred)

print(f"Stacking (blender) test accuracy: {stack_test_acc:.5f}")

# 4) Compare to the voting classifier from Exercise 8 (if you trained it)
try:
    vote_test_acc = accuracy_score(y_test, voting_soft.predict(X_test))
    print(f"Soft Voting test accuracy:        {vote_test_acc:.5f}")
    print(f"Stacking - Voting:                {stack_test_acc - vote_test_acc:+.5f}")
except NameError:
    print("voting_soft not found; train it in Exercise 8 to compare.")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Stacking (blender) test accuracy: 0.97260
Soft Voting test accuracy:        0.98150
Stacking - Voting:                -0.00890
